核心逻辑“四阶段闭环”：拾物入库、失主查询、混合检索、Agent 裁决与核销；同时保留“校园驿站/寄存点”这个很落地的业务设计。该系统的 V2.0 业务流：拾者拍照并选择存放点，经 Qwen-VL、CLIP、ChromaDB 入库；失者输入文本/图片后进行混合检索；DeepSeek Agent 复核并生成取件攻略；最终通过领取确认完成向量剔除和数据闭环。

---

# 基于多模态融合 RAG 与 Agent 的校园智能寻物系统项目书

## 一、项目名称

**基于多模态融合 RAG 与 Agent 的校园智能寻物系统**

也可作为展示型名称：

**基于视觉大模型与校园锚点网络的智能寻物 Agent 系统**

---

## 二、项目背景与建设必要性

在高校、园区、企业办公区等人员密集场景中，水杯、耳机、校园卡、钥匙、U 盘、雨伞、书包等小件物品丢失频率较高。传统失物招领方式主要依赖人工登记、微信群通知、公告栏发布或保安室人工保管，存在以下问题：

第一，**信息表达不统一**。拾物者上传图片、文字描述、人工登记表之间缺乏统一标准，同一物品可能被描述为“黑色杯子”“保温杯”“水壶”，导致检索困难。

第二，**图文匹配能力弱**。失主往往只能凭记忆输入文字描述，而系统中保存的多是图片或人工记录。传统关键词检索只能做字面匹配，难以支持“以文搜图”“以图搜图”和模糊描述匹配。

第三，**失物存放网点分散**。校园内失物可能分散在食堂前台、图书馆、保卫室、宿舍楼、教学楼值班室等多个寄存点。失主不知道该去哪里找，管理员也缺少统一入口。

第四，**认领流程缺少闭环**。多数失物招领系统只解决“发布”和“查看”，但没有解决“智能匹配—取件指引—状态核销—数据清理”的完整流转，导致已认领物品仍被重复检索，系统数据越来越脏。

因此，本项目拟建设一套面向校园场景的智能寻物系统，通过**视觉大模型、CLIP 多模态编码、ChromaDB 向量数据库、DeepSeek Agent 决策调度**等技术，实现失物从入库、检索、匹配、通知到核销的端到端闭环。

---

## 三、项目建设目标

本项目目标不是简单做一个“失物登记平台”，而是建设一个具备**多模态理解、混合检索、智能裁决、自动化流转**能力的企业级智能寻物系统。

具体目标包括：

1. **实现失物多模态智能入库**
   拾物者上传物品图片并选择存放驿站后，系统自动调用视觉大模型提取类别、颜色、品牌、Logo、破损细节、显著特征等结构化信息，并与图像向量、文本向量、地点、时间、状态等元数据统一入库。

2. **实现图文统一向量检索**
   支持失主通过文字描述或历史旧图发起寻物请求。系统将文本或图片统一映射到同一语义向量空间，实现“以文搜图”“以图搜图”和模糊语义匹配。

3. **实现预过滤混合检索机制**
   在大规模失物库中，先通过地点、时间、状态等 Metadata 硬约束过滤无效数据，再通过融合向量进行余弦相似度软匹配，提升召回准确率和检索效率。

4. **实现 Agent 智能裁决与取件指引**
   对 Top 候选结果，调用 DeepSeek Agent 基于失主描述和候选物 JSON 特征进行二次逻辑判断，输出匹配置信度、判断理由和取件攻略。

5. **实现数据生命周期闭环**
   失主确认领回后，系统自动将物品状态从“待认领”更新为“已认领”，并从向量库中删除或屏蔽对应向量，避免重复召回，保持系统长期健康运行。

---

## 四、总体设计思路

本项目采用“**多模态入库—统一查询—混合检索—Agent 裁决—核销闭环**”的总体架构。

整体流程如下：

**拾物者入库流：**

拾物者上传原图
→ 选择存放驿站
→ 系统记录上传时间
→ Qwen-VL 提取结构化 JSON
→ CLIP Image Encoder 提取图像向量
→ CLIP Text Encoder 提取文本向量
→ 图文向量加权融合
→ 存入 ChromaDB，并绑定地点、时间、状态、图片路径、JSON 特征等元数据。

**失主寻物流：**

失主输入文字描述或上传历史旧图
→ 系统生成统一查询向量
→ Metadata 硬过滤
→ 融合向量相似度检索
→ 召回 Top 5 候选物
→ DeepSeek Agent 二次裁决
→ 输出置信度和取件攻略
→ 用户前往驿站领取
→ 确认核销并删除向量。

该设计的关键价值在于：**前端输入可以不统一，但后端表达必须统一；入库数据可以是图片，查询可以是文字或图片，但最终都进入统一向量空间和统一业务闭环。**

---

## 五、系统总体架构

系统采用五层架构设计：

### 1. 用户交互层

面向三类角色：

**拾物者**：上传拾到物品照片，选择存放驿站，提交入库。
**失主**：输入物品描述或上传历史旧图，发起寻物请求，查看匹配结果和取件攻略。
**管理人员/驿站人员**：查看待认领物品、辅助确认领取、处理异常申诉。

MVP 阶段可采用 Streamlit 快速实现；正式部署阶段可升级为 Web 前端、小程序或校园 App 内嵌模块。

---

### 2. 多模态理解层

该层负责把非结构化图像转化为系统可计算、可检索、可推理的结构化数据。

核心能力包括：

* 调用 Qwen-VL 对失物图片进行视觉理解；
* 提取类别、颜色、品牌、材质、Logo、破损、特殊标记等信息；
* 将视觉细节转化为结构化 JSON；
* 对大模型输出进行格式校验和清洗，保证 JSON 可解析、可入库、可用于后续 Agent 推理。

示例 JSON：

```json
{
  "category": "水杯",
  "color": "黑色",
  "brand": "THERMOS",
  "material": "金属",
  "features": "杯身有银色 Logo，底部有轻微划痕",
  "condition": "轻微使用痕迹"
}
```

---

### 3. 多模态向量表征层

该层负责将图像和文本映射到统一向量空间。

核心设计包括：

* 使用 CLIP Image Encoder 将失物原图转化为图像向量；
* 使用 CLIP Text Encoder 将 Qwen-VL 输出的 JSON 文本转化为文本向量；
* 对图像向量和文本向量进行加权融合，形成“超级融合向量”；
* 入库时以融合向量作为主检索向量；
* 查询时根据输入类型生成统一查询向量。

融合方式可采用：

```text
Fused_Vector = 0.6 × Image_Vector + 0.4 × Text_Vector
```

其中，图像向量保留整体外观、形状、颜色等视觉信息；文本向量补充品牌、Logo、破损、特殊标记等细粒度语义信息。这样比单独使用图片向量更稳定，也比单独使用文本描述更具视觉容错能力。

---

### 4. 混合检索层

该层负责从失物库中高效召回候选物。

系统采用“双漏斗检索机制”：

#### 漏斗一：Metadata 硬约束过滤

通过地点、时间、状态等结构化字段先过滤不合理数据。

例如：

```text
WHERE 入库时间 >= 丢失时间
AND 状态 = '待认领'
AND 存放地点 IN 用户选择范围
```

该层解决的是“逻辑上不可能匹配”的问题。例如，用户昨天丢的物品，不应该匹配到三天前已经核销的物品。

#### 漏斗二：融合向量软匹配

在硬过滤后的候选池中，计算查询向量与库中融合向量之间的余弦相似度，取 Top K 候选物进入 Agent 裁决层。

该层解决的是“描述不完全一致但语义相近”的问题。例如，用户输入“黑色保温杯”，系统仍能召回“黑色膳魔师水杯”。

---

### 5. Agent 裁决与动作闭环层

该层负责对候选结果进行二次判断和业务动作触发。

ChromaDB 的向量检索擅长“相似召回”，但不擅长复杂逻辑判断。例如：

* 失主说“杯子没有贴纸”，候选物 JSON 显示“杯身有皮卡丘贴纸”；
* 失主说“白色耳机盒”，候选物是“白色鼠标”；
* 失主说“外壳无破损”，候选物显示“边角明显破裂”。

这些需要文本逻辑推理，因此系统引入 DeepSeek Agent 进行最终裁决。

Agent 输入：

```text
失主原始描述：
我昨天在二食堂丢了一个黑色膳魔师水杯，杯底有一点划痕，没有贴纸。

候选物 JSON：
{
  "category": "水杯",
  "color": "黑色",
  "brand": "THERMOS",
  "features": "杯底有轻微划痕，无明显贴纸",
  "location": "二食堂前台"
}
```

Agent 输出：

```json
{
  "confidence_score": 94,
  "match_result": "高度匹配",
  "reason": "类别、颜色、品牌、划痕细节和丢失地点均一致",
  "action": "notify_user",
  "pickup_guide": "该物品疑似为您丢失的水杯，目前存放在二食堂前台，请携带有效证件前往核对领取。"
}
```

前面材料里也强调了 DeepSeek Agent 的作用不是“重新看图”，而是基于 VLM 已经转化出的文本 JSON 做逻辑复核、置信度判断和取件指令生成。

---

## 六、核心业务流程设计

### 1. 拾物入库流程

1. 拾物者进入“我捡到了物品”页面；
2. 上传物品原图；
3. 选择存放驿站，例如“一食堂前台”“二食堂前台”“图书馆失物招领处”“东门保卫室”；
4. 系统记录当前时间；
5. Qwen-VL 提取结构化 JSON；
6. CLIP 提取图片向量和文本向量；
7. 系统融合向量并写入 ChromaDB；
8. 同步写入关系型数据库或本地记录表；
9. 返回入库成功结果和物品编号。

---

### 2. 失主查询流程

1. 失主进入“我丢失了物品”页面；
2. 输入自然语言描述，或上传历史旧图；
3. 可选填写丢失时间、可能丢失地点；
4. 系统生成查询向量；
5. ChromaDB 执行 Metadata 预过滤；
6. 对候选物执行向量相似度检索；
7. 返回 Top 5 候选物；
8. DeepSeek Agent 对候选物逐一裁决；
9. 前端展示匹配度、候选图片、取件地点、判断理由和取件攻略。

---

### 3. 认领核销流程

1. 失主根据取件攻略前往指定驿站；
2. 驿站工作人员人工核验物品；
3. 失主在系统中点击“我已领回”；
4. 系统将物品状态更新为“已认领”；
5. 系统从 ChromaDB 中删除该物品向量，或将其状态改为不可检索；
6. 生成核销记录，保留必要的审计信息；
7. 本次寻物流程结束。

前面方案中也明确提出了“极简版核销”：失主领回后点击“我已在存放点领回”，系统将状态改为 closed，并从向量库中移除 Embedding；这使系统不是简单推荐，而是形成完整生命周期闭环。

---

## 七、功能模块设计

### 模块一：拾物登记模块

**功能定位：** 负责失物数据采集和入库。

**主要功能：**

* 图片上传；
* 驿站地点选择；
* 当前时间自动记录；
* 物品描述自动生成；
* 入库结果展示；
* 物品编号生成。

**输入数据：**

* 原始图片；
* 存放地点；
* 拾物备注；
* 上传时间。

**输出数据：**

* item_id；
* image_path；
* VLM JSON；
* 融合向量；
* metadata；
* status = 待认领。

---

### 模块二：视觉结构化解析模块

**功能定位：** 负责将图片转成结构化语义。

**主要功能：**

* 识别物品类别；
* 提取主体颜色；
* 判断品牌或 Logo；
* 描述破损、划痕、贴纸、挂件等细节；
* 输出标准 JSON；
* 对异常输出进行格式修复。

**关键要求：**

* JSON 字段固定；
* 不允许输出大段自然语言；
* 对无法确定字段填“未知”；
* 输出结果需要可用于检索、展示和 Agent 推理。

---

### 模块三：多模态向量编码模块

**功能定位：** 负责统一图像、文本和融合向量表达。

**主要功能：**

* 图片向量编码；
* JSON 文本向量编码；
* 查询文本向量编码；
* 查询图片向量编码；
* 图文融合向量生成；
* 向量归一化处理。

**技术建议：**

* 中文场景优先采用支持中文的 CLIP 或中文多模态模型；
* MVP 可采用 sentence-transformers 生态中的多语言 CLIP；
* 后续可替换为 BGE-Visual、CN-CLIP、Qwen-CLIP 等模型。

---

### 模块四：向量数据库与元数据管理模块

**功能定位：** 负责失物向量和结构化元数据存储。

**核心字段设计：**

```json
{
  "item_id": "uuid",
  "image_path": "uploads/item_001.jpg",
  "category": "水杯",
  "color": "黑色",
  "brand": "THERMOS",
  "features": "杯底有划痕",
  "location": "二食堂前台",
  "upload_time": "2026-05-21 14:30:00",
  "status": "待认领"
}
```

**向量库字段：**

* id：物品唯一编号；
* embedding：超级融合向量；
* metadata：地点、时间、类别、状态、图片路径、JSON 文本；
* document：可存储 JSON 文本摘要。

---

### 模块五：混合检索模块

**功能定位：** 负责候选物召回。

**检索策略：**

1. 状态过滤：只检索“待认领”物品；
2. 时间过滤：只检索丢失时间之后入库的物品；
3. 地点过滤：优先检索用户选择区域；
4. 向量召回：计算查询向量与候选向量的相似度；
5. Top K 输出：默认输出 Top 5。

**优势：**

* 比关键词检索更能处理模糊描述；
* 比纯向量检索更能避免时空错配；
* 比纯大模型判断更快、更便宜、可规模化。

前面材料也把这一点概括为“向量语义检索 + 物理地点元数据过滤”的混合搜索机制，用来缓解单纯向量检索造成的误召回问题。

---

### 模块六：Agent 裁决模块

**功能定位：** 负责候选物二次精排和业务动作生成。

**主要功能：**

* 对比失主描述和候选物 JSON；
* 判断是否为同一物品；
* 输出 0-100 置信度；
* 生成判断理由；
* 生成取件攻略；
* 输出 action 指令。

**Action 示例：**

```json
{
  "action": "notify_user",
  "target_user": "lost_user_id",
  "item_id": "item_001",
  "message": "系统发现高度匹配物品，请前往二食堂前台核对领取。"
}
```

---

### 模块七：通知与取件引导模块

**功能定位：** 负责将 Agent 的判断结果转化为用户可执行动作。

**主要功能：**

* 展示候选物图片；
* 展示存放地点；
* 展示置信度；
* 展示取件攻略；
* 展示领取注意事项；
* 提供“我已领回”按钮。

**取件攻略示例：**

```text
系统发现与您描述高度匹配的物品，目前存放在【二食堂前台】。请携带校园卡或有效证件前往核对领取，领取后请点击“我已领回”完成核销。
```

---

### 模块八：数据生命周期管理模块

**功能定位：** 负责系统数据健康和闭环管理。

**主要功能：**

* 待认领状态管理；
* 已认领状态管理；
* 超期未领取提醒；
* 核销后向量删除；
* 历史审计记录保留；
* 异常认领申诉处理。

**推荐状态机：**

```text
待认领 → 已匹配 → 待领取 → 已认领 → 已核销
                    ↓
                 人工复核
```

---

## 八、技术路线

### 1. 前端技术路线

MVP 阶段：

* Streamlit；
* 文件上传组件；
* 下拉选择组件；
* 搜索结果卡片；
* 状态按钮；
* Session State 管理页面状态。

企业部署阶段：

* Vue / React；
* 小程序端；
* 校园 App 内嵌；
* 管理后台；
* 权限与角色管理。

---

### 2. 后端技术路线

MVP 阶段：

* Python；
* FastAPI 或 Streamlit 内部服务；
* 本地文件存储；
* ChromaDB；
* SQLite。

企业部署阶段：

* FastAPI / Flask；
* PostgreSQL / MySQL；
* MinIO / OSS 图片存储；
* ChromaDB / Milvus / Elasticsearch Vector；
* Redis 缓存；
* Docker 部署；
* Nginx 反向代理。

---

### 3. AI 模型技术路线

| 能力    | 推荐技术                | 作用            |
| ----- | ------------------- | ------------- |
| 图片理解  | Qwen-VL             | 提取结构化 JSON    |
| 图像编码  | CLIP Image Encoder  | 生成图片向量        |
| 文本编码  | CLIP Text Encoder   | 生成文本向量        |
| 向量检索  | ChromaDB            | 相似度召回         |
| 智能裁决  | DeepSeek            | 二次推理与取件攻略     |
| 工作流调度 | Python Orchestrator | 串联模型、检索、通知、核销 |

---

## 九、关键创新点

### 1. 从“图片上传”升级为“多模态结构化入库”

系统不是简单保存图片，而是将图片解析为 JSON、向量、Metadata 三类数据，兼顾展示、检索和推理。

---

### 2. 从“关键词搜索”升级为“统一图文向量空间检索”

失主可以输入文字，也可以上传历史旧图。系统底层统一转成向量进行匹配，突破传统关键词检索的限制。

---

### 3. 从“纯向量召回”升级为“硬约束 + 软匹配”的混合漏斗检索

系统先用地点、时间、状态过滤逻辑错误数据，再进行语义相似度匹配，既提升效率，也减少误报。

---

### 4. 从“相似推荐”升级为“Agent 智能裁决”

向量相似不等于真实匹配。系统引入 DeepSeek 对候选物进行二次文本推理，可以识别否定语义、细节冲突和逻辑矛盾。

---

### 5. 从“找到结果”升级为“业务生命周期闭环”

系统不仅告诉用户“可能是这个”，还提供取件攻略、核销按钮和向量删除机制，实现从数据入库到数据销毁的闭环流转。

---

## 十、系统接口设计

### 1. 拾物入库接口

```python
report_found_item(
    image_path: str,
    location: str,
    finder_note: str = None
) -> dict
```

**返回示例：**

```json
{
  "status": "success",
  "item_id": "item_001",
  "location": "二食堂前台",
  "features": {
    "category": "水杯",
    "color": "黑色",
    "brand": "THERMOS",
    "features": "杯底有轻微划痕"
  }
}
```

---

### 2. 寻物检索接口

```python
search_lost_item(
    query_text: str = None,
    query_image_path: str = None,
    lost_time: str = None,
    expected_location: str = None,
    top_k: int = 5
) -> dict
```

**返回示例：**

```json
{
  "status": "success",
  "candidates": [
    {
      "item_id": "item_001",
      "score": 0.91,
      "image_path": "uploads/item_001.jpg",
      "metadata": {
        "category": "水杯",
        "location": "二食堂前台",
        "status": "待认领"
      }
    }
  ]
}
```

---

### 3. Agent 裁决接口

```python
evaluate_match(
    loser_query: str,
    candidate_json: dict,
    location: str
) -> dict
```

**返回示例：**

```json
{
  "confidence_score": 94,
  "match_result": "高度匹配",
  "reason": "类别、颜色、品牌和划痕细节均一致",
  "pickup_guide": "请前往二食堂前台核对领取该物品。"
}
```

---

### 4. 物品核销接口

```python
confirm_pickup(
    item_id: str
) -> bool
```

**动作：**

* 更新关系数据库状态；
* 删除或屏蔽 ChromaDB 向量；
* 写入核销日志；
* 返回核销成功状态。

---

## 十一、数据库设计

### 1. 物品主表：lost_found_items

| 字段            | 类型       | 说明              |
| ------------- | -------- | --------------- |
| item_id       | varchar  | 物品唯一编号          |
| image_path    | varchar  | 图片存储路径          |
| category      | varchar  | 物品类别            |
| color         | varchar  | 主体颜色            |
| brand         | varchar  | 品牌              |
| features      | text     | 显著特征            |
| location      | varchar  | 存放驿站            |
| upload_time   | datetime | 入库时间            |
| status        | varchar  | 待认领/已匹配/已认领/已核销 |
| metadata_json | json     | 完整结构化信息         |
| created_at    | datetime | 创建时间            |
| updated_at    | datetime | 更新时间            |

---

### 2. 检索日志表：search_logs

| 字段                | 类型       | 说明         |
| ----------------- | -------- | ---------- |
| search_id         | varchar  | 查询编号       |
| query_text        | text     | 查询文本       |
| query_image_path  | varchar  | 查询图片       |
| expected_location | varchar  | 用户选择地点     |
| top_item_ids      | text     | 召回候选列表     |
| agent_result      | json     | Agent 判定结果 |
| created_at        | datetime | 查询时间       |

---

### 3. 核销记录表：pickup_records

| 字段              | 类型       | 说明               |
| --------------- | -------- | ---------------- |
| record_id       | varchar  | 核销记录编号           |
| item_id         | varchar  | 物品编号             |
| pickup_time     | datetime | 领取时间             |
| pickup_location | varchar  | 领取地点             |
| confirm_type    | varchar  | 用户确认/管理员确认/二维码确认 |
| remark          | text     | 备注               |

---

## 十二、实施计划

### 第一阶段：需求分析与系统设计

周期：1 周

主要工作：

* 梳理校园失物招领业务流程；
* 明确拾物者、失主、驿站管理员三类角色；
* 确定驿站地点字典；
* 设计系统总体架构；
* 设计数据表、向量库结构和接口规范；
* 绘制系统逻辑架构图。

交付成果：

* 项目需求说明；
* 总体架构图；
* 数据流图；
* 数据库设计；
* 接口设计文档。

---

### 第二阶段：核心能力开发

周期：2 周

主要工作：

* 接入 Qwen-VL 图片结构化解析；
* 封装 CLIP 图像/文本编码模块；
* 建设 ChromaDB 向量库；
* 实现图文融合向量入库；
* 实现 Metadata 过滤和向量检索；
* 实现 DeepSeek Agent 裁决模块。

交付成果：

* VLM 解析服务；
* 向量数据库管理模块；
* 混合检索模块；
* Agent 裁决模块；
* 核心后端工作流。

---

### 第三阶段：前端与业务闭环开发

周期：1 周

主要工作：

* 开发拾物端页面；
* 开发失主查询页面；
* 开发候选结果卡片；
* 展示候选图片、置信度、取件攻略；
* 开发“我已领回”核销按钮；
* 实现核销后状态更新和向量删除。

交付成果：

* 可运行前端系统；
* 拾物入库功能；
* AI 寻物功能；
* 核销闭环功能。

---

### 第四阶段：测试优化与展示验收

周期：1 周

主要工作：

* 构建测试数据集；
* 测试图文检索准确率；
* 测试地点过滤效果；
* 测试 Agent 判定稳定性；
* 优化 Prompt 和 JSON 清洗逻辑；
* 录制系统 Demo；
* 编写项目总结材料。

交付成果：

* 测试报告；
* Demo 视频；
* 项目说明文档；
* 部署说明；
* 验收材料。

---

## 十三、验收指标

### 1. 功能指标

| 指标       | 验收要求                    |
| -------- | ----------------------- |
| 拾物入库     | 支持图片上传、地点选择、自动解析和向量入库   |
| 图片解析     | 能输出类别、颜色、品牌、显著特征等结构化字段  |
| 文本寻物     | 支持失主通过自然语言描述检索候选物       |
| 图片寻物     | 支持失主上传历史旧图进行图图匹配        |
| 混合检索     | 支持地点、时间、状态等 Metadata 过滤 |
| Agent 裁决 | 能输出置信度、判断理由和取件攻略        |
| 核销闭环     | 认领后可删除或屏蔽对应向量           |

---

### 2. 性能指标

| 指标                | 建议目标                  |
| ----------------- | --------------------- |
| 单次入库处理时间          | 10 秒以内，取决于 VLM API 响应 |
| 单次检索响应时间          | 3 秒以内，不含 Agent 裁决     |
| Top 5 召回准确率       | 测试集上达到 80% 以上         |
| Agent 输出 JSON 成功率 | 95% 以上                |
| 核销成功率             | 100%                  |
| 重复召回率             | 已核销物品不再进入检索结果         |

---

### 3. 工程指标

| 指标    | 验收要求                           |
| ----- | ------------------------------ |
| 模块化程度 | VLM、向量库、Agent、Workflow、UI 分层封装 |
| 可替换性  | Qwen-VL、DeepSeek、CLIP 均可替换     |
| 可维护性  | 关键模块具备日志、异常处理、配置项              |
| 可扩展性  | 支持后续接入小程序、管理员端、二维码核销           |
| 数据安全  | 图片路径、用户记录、核销日志可追溯              |

---

## 十四、风险分析与应对措施

### 风险一：VLM 输出 JSON 不稳定

**表现：** 大模型可能输出 Markdown 代码块、自然语言解释或字段缺失。

**措施：**

* Prompt 中强制要求只输出 JSON；
* 后端使用正则清洗；
* 设置字段默认值；
* 对解析失败结果进行重试；
* 记录异常日志。

---

### 风险二：CLIP 对中文理解不足

**表现：** 原生 CLIP 偏英文，对中文描述检索效果不稳定。

**措施：**

* 采用多语言 CLIP 或中文 CLIP；
* 将用户中文描述先规范化；
* 用 VLM JSON 文本参与融合；
* 后续可切换为中文多模态模型。

---

### 风险三：纯向量检索误召回

**表现：** 外观相似但类别不同的物品被召回。

**措施：**

* 引入类别、地点、时间硬过滤；
* Top K 结果交给 Agent 二次裁决；
* 前端展示图片供用户二次确认；
* 对低置信度结果转人工处理。

---

### 风险四：误领或恶意领取

**表现：** 用户可能错误确认或恶意认领。

**措施：**

* 领取时保留图片与匹配编号；
* 可要求输入学生号或手机尾号；
* 关键物品由管理员确认；
* 后续支持二维码核销和人工复核。

---

### 风险五：API 调用成本与稳定性

**表现：** VLM 和 LLM 依赖外部 API，可能存在延迟或额度限制。

**措施：**

* 入库阶段才调用 VLM，避免频繁调用；
* 检索阶段优先使用本地 CLIP 和 ChromaDB；
* Agent 只对 Top 候选调用；
* 支持模型降级和缓存机制。

---

## 十五、项目成果形式

项目完成后形成以下成果：

1. **一套可运行的校园智能寻物系统原型**
   包括拾物入库、AI 寻物、候选展示、取件攻略、核销闭环等功能。

2. **一套多模态检索与 Agent 裁决后端代码**
   包括 Qwen-VL 解析、CLIP 编码、ChromaDB 检索、DeepSeek Agent 判断等核心模块。

3. **一套项目架构文档**
   包括系统架构图、业务流程图、数据流图、接口说明和数据库设计。

4. **一套测试数据与演示视频**
   使用真实物品图片构建测试集，展示从入库到查询再到核销的完整流程。

5. **可扩展部署方案**
   后续可升级为校园微信小程序、企业园区失物平台、图书馆/物业/保卫处智能失物系统。

---

## 十六、应用价值

### 1. 对用户

* 失主无需逐个群、逐个驿站询问；
* 通过文字或图片即可快速寻找；
* 系统直接给出候选图片和取件地点；
* 降低寻物时间和沟通成本。

### 2. 对校园管理方

* 统一管理分散失物信息；
* 减轻保安室、食堂、图书馆人工咨询压力；
* 提高失物认领效率；
* 形成可追溯的领取记录。

### 3. 对技术展示

该系统体现了完整的大模型应用工程能力：

* 多模态理解；
* 向量数据库；
* RAG 检索；
* Agent 决策；
* 工具调用；
* 数据生命周期管理；
* 前后端业务闭环。

它不是单点模型调用，而是一个具备企业级架构雏形的端到端智能业务系统。

---

## 十七、项目总结

本项目面向校园失物招领场景，构建了一套基于多模态大模型与混合 RAG 的智能寻物系统。系统以“校园驿站/寄存点”为业务锚点，将拾物者、失主和存放点纳入统一流程，通过 Qwen-VL 完成图片结构化解析，通过 CLIP 完成图文统一向量表达，通过 ChromaDB 实现带 Metadata 预过滤的高效混合检索，并通过 DeepSeek Agent 完成候选物精排裁决和取件攻略生成。

相比传统失物招领系统，本项目的核心提升在于：**从人工登记变为智能入库，从关键词搜索变为图文语义检索，从相似推荐变为 Agent 裁决，从信息发布变为领取核销闭环。**

最终系统能够实现：

```text
数据入库 → 多模态理解 → 融合向量检索 → Agent 智能裁决 → 用户取件 → 数据核销
```

这使项目既具备清晰的工程落地路径，也具备较强的大模型应用创新性和企业级系统设计价值。
